In [39]:
import ROOT
import pandas as pd
import numpy as np
import os
from pathlib import Path
import matplotlib.pyplot as plt

# === Configuration ===
year = "2016APV"
bgr = "signal"
folder = f"plots/{bgr}/{year}/root"

# === Dictionary to hold all data in memory ===
tables = {}

# === List ROOT files ===
root_files = [f for f in os.listdir(folder) if f.endswith('.root')]
print(f"Processing {len(root_files)} ROOT files...")

for filename in root_files:
    print(f"\nProcessing: {filename}")
    file_path = f"{folder}/{filename}"

    try:
        file = ROOT.TFile.Open(file_path)
        if not file or file.IsZombie():
            print(f"  ✗ Skipping invalid file: {filename}")
            continue

        # --- Collect histograms ---
        histograms = []
        for key in file.GetListOfKeys():
            obj = key.ReadObj()
            if obj and obj.InheritsFrom("TH1"):
                histograms.append(obj)

        if not histograms:
            file.Close()
            print(f"  ⚠️ No histograms found in {filename}")
            continue

        # --- Build a table (DataFrame) with binning information ---
        first_hist = histograms[0]
        nbins = first_hist.GetNbinsX()
        bin_data = {
            "bin_number": np.arange(1, nbins + 1),
            "low_edge": [first_hist.GetBinLowEdge(i) for i in range(1, nbins + 1)],
            "center": [first_hist.GetBinCenter(i) for i in range(1, nbins + 1)],
            "high_edge": [first_hist.GetBinLowEdge(i) + first_hist.GetBinWidth(i) for i in range(1, nbins + 1)],
            "width": [first_hist.GetBinWidth(i) for i in range(1, nbins + 1)],
        }

        # Add each histogram as a column
        for hist in histograms:
            contents = [hist.GetBinContent(i) for i in range(1, nbins + 1)]
            bin_data[hist.GetName()] = contents

        df = pd.DataFrame(bin_data)

        # Store the DataFrame in the dictionary
        tables[filename] = df

        print(f"  ✓ Loaded {len(histograms)} histograms with {nbins} bins")

        file.Close()

    except Exception as e:
        print(f"  ✗ Error processing {filename}: {str(e)}")

print("\n✅ All ROOT files loaded in memory.")
print("You can access them with:")
print("  tables['<filename>.root']")

# === Optional: helper function to visualize a histogram ===
def show_histogram(file_name, hist_name):
    """
    Plot a histogram from the in-memory table.

    Args:
        file_name (str): ROOT filename key in `tables`.
        hist_name (str): Column name of the histogram to plot.
    """
    if file_name not in tables:
        print(f"⚠️ File '{file_name}' not found in tables.")
        return
    df = tables[file_name]

    if hist_name not in df.columns:
        print(f"⚠️ Histogram '{hist_name}' not found in {file_name}.")
        print("Available histograms:", [c for c in df.columns if c not in ['bin_number','low_edge','center','high_edge','width']])
        return

    plt.figure(figsize=(8, 5))
    plt.bar(df["center"], df[hist_name], width=df["width"], align="center", alpha=0.7)
    plt.xlabel("X-axis (variable)")
    plt.ylabel("Entries")
    plt.title(f"{hist_name} — {file_name}")
    plt.grid(True, alpha=0.3)
    plt.show()


Processing 17 ROOT files...

Processing: DYJetsToLNu_signal_tau_2016APV.root
  ✓ Loaded 59 histograms with 9 bins

Processing: Diboson_signal_tau_2016APV.root
  ✓ Loaded 59 histograms with 9 bins

Processing: Higgs_signal_tau_2016APV.root
  ✓ Loaded 59 histograms with 9 bins

Processing: QCD_signal_tau_2016APV.root
  ✓ Loaded 73 histograms with 9 bins

Processing: SignalTau_1000GeV_signal_tau_2016APV.root
  ✓ Loaded 59 histograms with 9 bins

Processing: SignalTau_1500GeV_signal_tau_2016APV.root
  ✓ Loaded 59 histograms with 9 bins

Processing: SignalTau_2000GeV_signal_tau_2016APV.root
  ✓ Loaded 59 histograms with 9 bins

Processing: SignalTau_3000GeV_signal_tau_2016APV.root
  ✓ Loaded 59 histograms with 9 bins

Processing: SignalTau_300GeV_signal_tau_2016APV.root
  ✓ Loaded 59 histograms with 9 bins

Processing: SignalTau_400GeV_signal_tau_2016APV.root
  ✓ Loaded 59 histograms with 9 bins

Processing: SignalTau_600GeV_signal_tau_2016APV.root
  ✓ Loaded 59 histograms with 9 bins

Proc

In [40]:
from IPython.display import display  # For pretty DataFrame display in notebooks

for root_name in list(tables.keys()):
    df = tables[root_name]
    
    print("==================================================================")
    print(f"📁 ROOT file: {root_name}")
    
    # --- List all available histograms in the file ---
    all_hists = [c for c in df.columns 
                 if c not in ['bin_number', 'low_edge', 'center', 'high_edge', 'width']]
    
    print("📜 Available histograms:")
    for h in all_hists:
        print(f"   • {h}")
    
    # --- Find the histogram that ends with 'nom' ---
    hist_candidates = [c for c in all_hists if c.endswith("nom")]
    
    if not hist_candidates:
        print(f"⚠️ No histogram ending with 'nom' found in {root_name}.\n")
        continue
    
    # Use the first histogram found that matches the 'nom' pattern
    hist_name = hist_candidates[0]
    
    # --- Extract histogram info ---
    nbins = len(df)
    xmin = df["low_edge"].min()
    xmax = df["high_edge"].max()
    
    print(f"\n📊 Selected histogram (ends with 'nom'): {hist_name}")
    print(f"   • Number of bins: {nbins}")
    print(f"   • Range: [{xmin:.2f}, {xmax:.2f}]\n")
    
    # --- Add a total row at the bottom ---
    total_value = df[hist_name].sum()
    total_row = {
        "low_edge": "Total",
        "high_edge": "",
        hist_name: total_value
    }
    df_with_total = pd.concat([df[["low_edge", "high_edge", hist_name]], pd.DataFrame([total_row])], ignore_index=True)
    
    # --- Display the histogram data as a pandas table with total row ---
    display(df_with_total)



📁 ROOT file: DYJetsToLNu_signal_tau_2016APV.root
📜 Available histograms:
   • DYJetsToLNu_signal_tau_2016APV_nom
   • CMS_rochester_signal_tau_2016APV_Up
   • CMS_t_energy_signal_tau_2016APV_Up
   • CMS_scale_j_signal_tau_2016APV_Up
   • CMS_res_j_signal_tau_2016APV_Up
   • CMS_MET_unclustered_signal_tau_2016APV_Up
   • CMS_scale_fj_signal_tau_2016APV_Up
   • CMS_res_fj_signal_tau_2016APV_Up
   • CMS_l1_ecal_prefiring_signal_tau_2016APV_Up
   • ps_ISR_signal_tau_2016APV_Up
   • ps_FSR_signal_tau_2016APV_Up
   • PDF_signal_tau_2016APV_Up
   • Alpha(PDF)_signal_tau_2016APV_Up
   • CMS_pileup_signal_tau_2016APV_Up
   • CMS_eff_j_PUJET_signal_tau_2016APV_Up
   • CMS_btag_heavy_signal_tau_2016APV_Up
   • CMS_btag_light_signal_tau_2016APV_Up
   • CMS_eff_e_id_signal_tau_2016APV_Up
   • CMS_eff_e_reco_above20_signal_tau_2016APV_Up
   • CMS_eff_e_reco_below20_signal_tau_2016APV_Up
   • CMS_eff_m_reco_signal_tau_2016APV_Up
   • CMS_eff_m_id_signal_tau_2016APV_Up
   • CMS_eff_m_iso_signal_tau_20

,low_edge,high_edge,DYJetsToLNu_signal_tau_2016APV_nom
0,0.0,20.0,23.013720
1,20.0,40.0,17.234488
2,40.0,60.0,7.383359
3,60.0,80.0,-1.529812
4,80.0,100.0,0.000000
5,100.0,120.0,-1.595932
6,120.0,150.0,0.000000
7,150.0,200.0,0.000000
8,200.0,300.0,0.000000
9,Total,,44.505822


📁 ROOT file: Diboson_signal_tau_2016APV.root
📜 Available histograms:
   • Diboson_signal_tau_2016APV_nom
   • CMS_rochester_signal_tau_2016APV_Up
   • CMS_t_energy_signal_tau_2016APV_Up
   • CMS_scale_j_signal_tau_2016APV_Up
   • CMS_res_j_signal_tau_2016APV_Up
   • CMS_MET_unclustered_signal_tau_2016APV_Up
   • CMS_scale_fj_signal_tau_2016APV_Up
   • CMS_res_fj_signal_tau_2016APV_Up
   • CMS_l1_ecal_prefiring_signal_tau_2016APV_Up
   • ps_ISR_signal_tau_2016APV_Up
   • ps_FSR_signal_tau_2016APV_Up
   • PDF_signal_tau_2016APV_Up
   • Alpha(PDF)_signal_tau_2016APV_Up
   • CMS_pileup_signal_tau_2016APV_Up
   • CMS_eff_j_PUJET_signal_tau_2016APV_Up
   • CMS_btag_heavy_signal_tau_2016APV_Up
   • CMS_btag_light_signal_tau_2016APV_Up
   • CMS_eff_e_id_signal_tau_2016APV_Up
   • CMS_eff_e_reco_above20_signal_tau_2016APV_Up
   • CMS_eff_e_reco_below20_signal_tau_2016APV_Up
   • CMS_eff_m_reco_signal_tau_2016APV_Up
   • CMS_eff_m_id_signal_tau_2016APV_Up
   • CMS_eff_m_iso_signal_tau_2016APV_Up

,low_edge,high_edge,Diboson_signal_tau_2016APV_nom
0,0.0,20.0,4.028414
1,20.0,40.0,4.270422
2,40.0,60.0,2.558676
3,60.0,80.0,1.028183
4,80.0,100.0,0.578063
5,100.0,120.0,0.310534
6,120.0,150.0,0.367172
7,150.0,200.0,0.364748
8,200.0,300.0,0.280712
9,Total,,13.786924


📁 ROOT file: Higgs_signal_tau_2016APV.root
📜 Available histograms:
   • Higgs_signal_tau_2016APV_nom
   • CMS_rochester_signal_tau_2016APV_Up
   • CMS_t_energy_signal_tau_2016APV_Up
   • CMS_scale_j_signal_tau_2016APV_Up
   • CMS_res_j_signal_tau_2016APV_Up
   • CMS_MET_unclustered_signal_tau_2016APV_Up
   • CMS_scale_fj_signal_tau_2016APV_Up
   • CMS_res_fj_signal_tau_2016APV_Up
   • CMS_l1_ecal_prefiring_signal_tau_2016APV_Up
   • ps_ISR_signal_tau_2016APV_Up
   • ps_FSR_signal_tau_2016APV_Up
   • PDF_signal_tau_2016APV_Up
   • Alpha(PDF)_signal_tau_2016APV_Up
   • CMS_pileup_signal_tau_2016APV_Up
   • CMS_eff_j_PUJET_signal_tau_2016APV_Up
   • CMS_btag_heavy_signal_tau_2016APV_Up
   • CMS_btag_light_signal_tau_2016APV_Up
   • CMS_eff_e_id_signal_tau_2016APV_Up
   • CMS_eff_e_reco_above20_signal_tau_2016APV_Up
   • CMS_eff_e_reco_below20_signal_tau_2016APV_Up
   • CMS_eff_m_reco_signal_tau_2016APV_Up
   • CMS_eff_m_id_signal_tau_2016APV_Up
   • CMS_eff_m_iso_signal_tau_2016APV_Up
   

...

,low_edge,high_edge,WJetToLNu_signal_tau_2016APV_nom
0,0.0,20.0,249.620987
1,20.0,40.0,212.613861
2,40.0,60.0,137.714233
3,60.0,80.0,80.279343
4,80.0,100.0,20.226593
5,100.0,120.0,6.224182
6,120.0,150.0,4.968804
7,150.0,200.0,2.502737
8,200.0,300.0,13.899734
9,Total,,728.050475


📁 ROOT file: data_obs_signal_tau_2016APV.root
📜 Available histograms:
   • data_obs_signal_tau_2016APV_nom

📊 Selected histogram (ends with 'nom'): data_obs_signal_tau_2016APV_nom
   • Number of bins: 9
   • Range: [0.00, 300.00]



,low_edge,high_edge,data_obs_signal_tau_2016APV_nom
0,0.0,20.0,0.0
1,20.0,40.0,0.0
2,40.0,60.0,0.0
3,60.0,80.0,0.0
4,80.0,100.0,0.0
5,100.0,120.0,0.0
6,120.0,150.0,0.0
7,150.0,200.0,0.0
8,200.0,300.0,0.0
9,Total,,0.0


📁 ROOT file: tt_signal_tau_2016APV.root
📜 Available histograms:
   • tt_signal_tau_2016APV_nom
   • CMS_rochester_signal_tau_2016APV_Up
   • CMS_t_energy_signal_tau_2016APV_Up
   • CMS_scale_j_signal_tau_2016APV_Up
   • CMS_res_j_signal_tau_2016APV_Up
   • CMS_MET_unclustered_signal_tau_2016APV_Up
   • CMS_scale_fj_signal_tau_2016APV_Up
   • CMS_res_fj_signal_tau_2016APV_Up
   • CMS_l1_ecal_prefiring_signal_tau_2016APV_Up
   • ps_ISR_signal_tau_2016APV_Up
   • ps_FSR_signal_tau_2016APV_Up
   • PDF_signal_tau_2016APV_Up
   • Alpha(PDF)_signal_tau_2016APV_Up
   • CMS_pileup_signal_tau_2016APV_Up
   • CMS_eff_j_PUJET_signal_tau_2016APV_Up
   • CMS_btag_heavy_signal_tau_2016APV_Up
   • CMS_btag_light_signal_tau_2016APV_Up
   • CMS_eff_e_id_signal_tau_2016APV_Up
   • CMS_eff_e_reco_above20_signal_tau_2016APV_Up
   • CMS_eff_e_reco_below20_signal_tau_2016APV_Up
   • CMS_eff_m_reco_signal_tau_2016APV_Up
   • CMS_eff_m_id_signal_tau_2016APV_Up
   • CMS_eff_m_iso_signal_tau_2016APV_Up
   • CMS_

,low_edge,high_edge,tt_signal_tau_2016APV_nom
0,0.0,20.0,548.773254
1,20.0,40.0,546.149170
2,40.0,60.0,403.569702
3,60.0,80.0,234.518433
4,80.0,100.0,100.384834
5,100.0,120.0,52.770599
6,120.0,150.0,76.488953
7,150.0,200.0,99.016800
8,200.0,300.0,137.110062
9,Total,,2198.781807
